<!-- CHECK_ALLOWLIST: descriptive-posture banner cell - spec §9 / §7-quoting block, exempt per plan 0.13 -->
# E10 GSPS - 03 Surface Characterization (Phase 4 / task 4.4)

DESCRIPTIVE POSTURE - this notebook reports CALIBRATION-CONDITIONAL DESCRIPTIVE content; no inferential beta claim, no p-value, no hypothesis-rejection language, no statistical-test framing. Per spec v0.7 §9 the E10 deliverable is the FX-variance-share sensitivity surface (a description across the anchored Q-volume range), not a verdict point. The §7 field-1 sign gate is a necessary-not-sufficient descriptive gate, not an inferential test. The 5-currency cross-spread is a non-statistical display (NOT a confidence interval, NOT an inferential band, NOT a hypothesis test) - per CORR-E10P-11 / spec v0.6 W-2 the bootstrap and permutation arms were dropped at G=5 and replaced with the raw min-max / IQR envelope here. <!-- CHECK_ALLOWLIST -->

Scope (plan v0.2 task 4.4): consume the Phase-4 module results from `simulations/e10_gsps/modules/{vol_on_vol_regression,surface_grid,currency_spread}.py` and exhibit, with a trio HALT-checkpoint at each step:

1. the §4.1 vol-on-vol descriptive sign gate under both co-primary FE specs (task 4.1);
2. the §4.3 FX-variance-share sensitivity surface on the user-locked log-spaced 50-point grid across the anchored Q-range (task 4.2);
3. the user-locked interior-crossing scan (>=3 contiguous strictly-interior cells; task 4.2a);
4. the `q_variance_dominance_flag` - the spec §9 NON-RETIREMENT-rung input;
5. the non-statistical 5-currency min-max / IQR spread display with the mandatory in-figure firewall label (task 4.3);
6. the surface-vs-break-even comparison against the Phase-2.5-frozen s_be = 0.25;
7. the four §9-classifier flags the Phase-6 verdict step will consume.

Spec source: `docs/specs/2026-05-20-e10-gsps-v0.7-convex-multicurrency-design.md` (§4, §7, §9). Plan source: `docs/plans/2026-05-20-e10-gsps-v0.4-implementation-plan.md` (task 4.4 line 357). Phase-2.5 break-even record: `notebooks/e10_gsps/diagnostics/E10.2.5_break_even_threshold.md`. Phase-4 completion memo: `scratch/2026-05-20-e10-revision/e10_phase4_completion.md`.

## Trio 1 - setup + frozen break-even share

Why: load the Tier-1 panel and the three Phase-4 module entry points, then restate the Phase-2.5-frozen break-even share s_be = 0.25 that every surface-vs-threshold comparison in this notebook reads against. A HALT-checkpoint for human review per `memory/feedback_notebook_trio_checkpoint.md` - the trio confirms (a) the 150-cell panel loads cleanly, (b) the three module imports succeed, and (c) s_be is restated from the frozen Phase-2.5 record before any surface call.

### Decision-citation block - frozen break-even threshold s_be = 0.25

> Reference: `notebooks/e10_gsps/diagnostics/E10.2.5_break_even_threshold.md` (Phase-2.5 record); spec v0.7 §4.4 (ex-ante-pinned break-even share); plan v0.2 pre-pin Field 3 (LOCKED).
> Why: the break-even share is the threshold the §4.3 surface is read against; pinning it ex ante is the load-bearing anti-fishing safeguard - no post-hoc tuning of the threshold can flip the §9 classifier outcome.
> Relevance: s_be = 0.25 is the canonical horizontal reference line on every surface figure in this notebook, and the input that drives the `q_variance_dominance_flag` and the interior-crossing scan.
> Connection: consumed by `evaluate_surface_grid(..., s_be=0.25)`, `compute_currency_spread(..., s_be=0.25)`, and `detect_interior_crossing(grid, s_be=0.25)` - the three Phase-4 module calls this notebook exhibits.

In [1]:
# Trio discipline: every (why-markdown, code-cell, interpretation-
# markdown) trio is a HALT-checkpoint for human review per
# feedback_notebook_trio_checkpoint.md.
from __future__ import annotations

import json
import math
import statistics
import sys
from collections import defaultdict
from pathlib import Path

import matplotlib
matplotlib.use('Agg')  # headless execution (plan task 0.9)
import matplotlib.pyplot as plt
import numpy as np

REPO = Path.cwd()
while not (REPO / 'simulations').is_dir() and REPO != REPO.parent:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from simulations.e10_gsps.utils.panel_io import PanelParquetIO
from simulations.e10_gsps.modules.vol_on_vol_regression import run_vol_on_vol
from simulations.e10_gsps.modules.surface_grid import (
    evaluate_surface_grid,
    detect_interior_crossing,
)
from simulations.e10_gsps.modules.currency_spread import compute_currency_spread

PANEL = REPO / 'data' / 'panels' / 'e10_gsps_panel.parquet'
if not PANEL.exists():
    import subprocess
    subprocess.run(
        [sys.executable, str(REPO / 'scripts' / 'build_e10_gsps_panel.py')],
        check=True,
    )

cells_panel = PanelParquetIO(PANEL).read()
by_cur = defaultdict(list)
for c in cells_panel:
    by_cur[c.currency].append(c)

# Phase-2.5-frozen break-even share (E10.2.5_break_even_threshold.md).
S_BE = 0.25

# Anchored Q-volume range (spec v0.7 §6.2 - the E10.1 calibration
# note section 2 ~60-130 priceable-queries/month bracket; Q_low
# carried forward at 28 as the surviving-day floor per the Phase-3
# panel decomposition).
Q_RANGE = (28.0, 130.0)

print('panel cells           :', len(cells_panel))
print('currencies            :', sorted(by_cur))
print('months per currency   :', len(cells_panel) // len(by_cur))
print('material-gap cells    :', sum(1 for c in cells_panel if c.material_gap))
print('frozen s_be           :', S_BE)
print('anchored Q-range      :', Q_RANGE)

panel cells           : 150
currencies            : ['BRL', 'COP', 'EUR', 'GBP', 'NGN']
months per currency   : 30
material-gap cells    : 0
frozen s_be           : 0.25
anchored Q-range      : (28.0, 130.0)


### Interpretation

The 150-cell panel loads cleanly (5 currencies x 30 months, zero material-gap cells), and the three Phase-4 module entry points import cleanly. The break-even share is restated from the Phase-2.5 record at s_be = 0.25 - frozen ex ante, no post-hoc tuning. The anchored Q-range (Q_low=28, Q_high=130) is the E10.1-calibrated bracket the surface is evaluated across. No HALT.

## Trio 2 - vol-on-vol descriptive sign gate (task 4.1)

Why: the §4.1 vol-on-vol regression is the necessary-not-sufficient descriptive sign gate of spec §7 field 1 - regress the Y-side cost-stream realized variance on the X-side FX realized variance under the two co-primary FE specifications (two-way FE = currency + month; currency-FE-only) and report the descriptive sign of beta on each. The two-way-vs-currency-only gap is reported as descriptive content (a Frisch-Waugh-Lovell decomposition of global-vs-idiosyncratic FX-vol exposure); the gap is load-bearing for the spec §13 M-sketch design step that the Phase-6 verdict step routes to under SURFACE-PRODUCED. NO p-value, NO t-stat, NO inferential claim - at G=5 the cluster-robust variance estimator is structurally size-distorted and the G-caveat is reported alongside any point estimate.

### Decision-citation block - two co-primary FE specifications

> Reference: spec v0.7 §4.1; CORR-E10P-11 / spec v0.6 W-1 (the currency-FE-only arm promoted from a sensitivity arm to a co-primary descriptive specification).
> Why: the two-way FE absorbs global FX-vol co-movement (month FE) and currency baselines (currency FE), leaving the residualized idiosyncratic (within-month, cross-currency) variation; the currency-FE-only arm preserves the cross-month / cross-currency level information the §13 M-sketch hedge-sizing depends on. Reporting both prevents either specification from silently dominating the descriptive sign read.
> Relevance: the gap = beta_two_way - beta_currency_only quantifies how much of the beta signal lives in global vs idiosyncratic FX-vol exposure - the descriptive content load-bearing for the §13 M-sketch.
> Connection: feeds the spec §9 ladder's §7-field-1 sign gate (both signs must be +1 for SURFACE-PRODUCED; spec §9 rung classifier consumes the descriptive signs, not a t-stat or p-value).

In [2]:
# Run the §4.1 vol-on-vol regression under both co-primary FE specs.
vov = run_vol_on_vol(cells_panel)

print('=== §4.1 vol-on-vol descriptive sign gate ===')
print('two-way FE (currency + month)  beta =', format(vov.beta_two_way, '.6f'), ' sign =', vov.sign_two_way)
print('currency-FE-only               beta =', format(vov.beta_currency_only, '.6f'), ' sign =', vov.sign_currency_only)
print('Frisch-Waugh-Lovell gap        =', format(vov.gap, '.6f'))
print()
print('panel cells           :', vov.n_cells)
print('currency clusters (G) :', vov.n_currency_clusters)
print('residual DoF two-way  :', vov.residual_dof_two_way)
print('residual DoF cur-only :', vov.residual_dof_currency_only)
print()
print('G-caveat (descriptive context only):')
print(' ', vov.g_caveat_label)

=== §4.1 vol-on-vol descriptive sign gate ===
two-way FE (currency + month)  beta = 68.151246  sign = 1
currency-FE-only               beta = 54.714823  sign = 1
Frisch-Waugh-Lovell gap        = 13.436423

panel cells           : 150
currency clusters (G) : 5
residual DoF two-way  : 115
residual DoF cur-only : 144

G-caveat (descriptive context only):
  G~=5 currency clusters — descriptive sign only; clustered SE is structurally size-distorted at this G and is reported (if at all) as context, not as an inferential claim.


### Interpretation

The descriptive sign of beta under BOTH co-primary FE specifications is +1: beta_two_way = +68.151246 and beta_currency_only = +54.714823. The §4.1 necessary-not-sufficient descriptive sign gate holds on the spec §7 field 1 criterion - both signs point the predicted way.

The Frisch-Waugh-Lovell gap = +13.436423 is the descriptive content the §13 M-sketch design step depends on: the global (month-FE-absorbed) FX-vol co-movement contributes the larger share of the cross-cell beta signal (the two-way slope is larger than the currency-only slope), so a multi-currency convex hedge should size its cross-currency exposure with the global-vs-idiosyncratic split visible in this gap. This is descriptive content, not an inferential claim.

G-caveat (mandatory at G=5): the cluster-robust variance estimator is structurally size-distorted at this cluster count (the (G-1)/G Bessel-style correction is meaningless), so no clustered standard error or test statistic is reported in this notebook. The descriptive sign is the only object the §7-field-1 gate reads.

## Trio 3 - FX-variance-share surface on the user-locked log-spaced grid (task 4.2)

Why: the §4.3 deliverable is the FX-variance share Var(d log FX) / Var(d log cost) reported as a sensitivity surface across the WHOLE anchored Q-volume range, not at the endpoints. The user-locked grid resolution is log-spaced 50 points with adaptive doubling to 100 if any grid cell sits within +/-0.5 dex of s_be = 0.25 (decision locked 2026-05-21). The surface is the calibration-conditional descriptive object the Phase-6 §9 classifier reads.

### Decision-citation block - user-locked log-spaced 50-point grid

> Reference: pre-Phase-4 Model QA review item 2 (recommended default); user lock 2026-05-21; spec v0.7 §4.3 / plan task 4.2.
> Why: log-spacing catches the Q-low regime where Var(d log Q) collapses fastest and a linear grid would under-sample; 50 points = ~50 samples per dex on the ~0.5-1 dex anchored range, comparable to standard 1-D sensitivity-surface density; adaptive doubling to 100 closes the narrow-band-miss risk near s_be without committing 500+ points up front.
> Relevance: too-coarse a grid silently drifts the §9 verdict-rung classification (a missed interior crossing flips SURFACE-PRODUCED to a SURFACE-PRODUCED-with-stated-crossing classification).
> Connection: this grid is the substrate the §4.3 surface and the §9 verdict ladder are computed on; the same Q-grid is reused in Trio 5 (interior-crossing scan) and Trio 6 (5-currency spread display).

In [3]:
# Evaluate the §4.3 surface on the user-locked grid.
sg = evaluate_surface_grid(cells_panel, q_range=Q_RANGE, s_be=S_BE)

q_grid = [pt.q_volume for pt in sg.points]
shares = [pt.fx_variance_share for pt in sg.points]

print('=== §4.3 FX-variance-share surface ===')
print('anchored Q-range      :', (sg.anchored_range_low, sg.anchored_range_high))
print('effective_n_grid      :', sg.effective_n_grid)
print('refined (doubled)     :', sg.refined)
print('log-step (dex)        :', format(sg.grid_resolution, '.4f'))
print('break-even s_be       :', sg.break_even_share)
print()
print('share at Q_low  ({:.1f}) :'.format(q_grid[0]), format(shares[0], '.6e'))
print('share at Q_med  ({:.1f}) :'.format(q_grid[len(q_grid) // 2]),
      format(shares[len(shares) // 2], '.6e'))
print('share at Q_high ({:.1f}):'.format(q_grid[-1]), format(shares[-1], '.6e'))
print('share min / max across grid:', format(min(shares), '.6e'),
      '/', format(max(shares), '.6e'))

# Plot: panel-mean surface vs Q-volume on log-log scale, with s_be reference line.
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(q_grid, shares, color='black', linewidth=2.0, label='panel-mean share')
ax.axhline(S_BE, color='red', linestyle='--', linewidth=1.5,
           label='s_be = 0.25 (Phase-2.5 frozen)')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('monthly Q-volume (log-scale)')
ax.set_ylabel('FX-variance share  Var(dlogFX) / Var(dlogcost)  (log-scale)')
ax.set_title('E10 §4.3 surface - DESCRIPTIVE; calibration-conditional only')
ax.legend(loc='center right')
ax.grid(True, which='both', linestyle=':', alpha=0.4)
plt.tight_layout()
plt.show()

=== §4.3 FX-variance-share surface ===
anchored Q-range      : (28.0, 130.0)
effective_n_grid      : 50
refined (doubled)     : False
log-step (dex)        : 0.0136
break-even s_be       : 0.25

share at Q_low  (28.0) : 1.535158e-04
share at Q_med  (61.3) : 1.535158e-04
share at Q_high (130.0): 1.535158e-04
share min / max across grid: 1.535158e-04 / 1.535158e-04


/tmp/ipykernel_64677/4048989681.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Interpretation

The §4.3 surface evaluates on the user-locked log-spaced 50-point grid across the anchored Q-range [28, 130]; the adaptive doubling rule did NOT fire (refined = False, effective_n_grid = 50) because no grid cell sits within +/-0.5 dex of s_be = 0.25 - the panel-mean share is approximately 1.5e-4 across the whole grid, roughly three decades below s_be. The surface is approximately invariant in Q-volume across the anchored range, which is what the Phase-3 panel-decomposition headline (mean share ~2e-4, max ~4e-3) anticipated.

Descriptive read: the FX-variance share is small everywhere on the anchored Q-volume range; Q-variance dominates the cost-stream variance by three-to-four orders of magnitude. This is the calibration-conditional surface, not an inferential claim; the §9 classifier (Phase 6) reads the qualitative shape together with the `q_variance_dominance_flag` emitted next.

## Trio 4 - `q_variance_dominance_flag` (spec §7 field 7b / §9 NON-RETIREMENT input)

Why: per spec v0.7 §7 field 7b, panel-wide Q-variance dominance is a spec-anticipated, valid, acceptable outcome that maps to the NON-RETIREMENT rung of the §9 descriptive ladder. The flag is emitted by `evaluate_surface_grid` and consumed by the Phase-6 §9 classifier (plan task 6.1). It is TRUE iff Var(d log Q) / Var(d log cost) > (1 - s_be) = 0.75 at every grid point - equivalently iff the FX-variance share is strictly below s_be at every grid point.

### Decision-citation block - `q_variance_dominance_flag` semantics

> Reference: pre-Phase-4 Model QA review item 6 (the flag was added to `SurfaceGridResult` so the §9 classifier consumes one well-typed boolean rather than a re-derived numerical check); spec v0.7 §7 field 7b (Q-variance dominance is an acceptable outcome).
> Why: a single well-typed boolean shields the §9 classifier from numerical drift in the grid-evaluation step; the flag's definition (share < s_be at every grid point) is the spec's literal NON-RETIREMENT criterion.
> Relevance: HALT-Q-DOMINANCE is one of the spec §7 field 7 HALT conditions - it is NOT a failure, it is a valid descriptive outcome the §9 classifier routes explicitly.
> Connection: consumed by Phase 6 task 6.1 alongside `interior_crossing`, `material_gap_flag`, and `surface_computed_flag` to produce the §9 verdict.

In [4]:
# Emit the §9-classifier inputs.
print('=== §9 NON-RETIREMENT-rung input ===')
print('panel-mean share (grid-invariant) :', format(shares[0], '.6e'))
print('s_be threshold                    :', S_BE)
print('1 - s_be (Q-dominance threshold)  :', 1.0 - S_BE)
print('share / s_be (decades below)      :', format(math.log10(shares[0] / S_BE), '.2f'), 'dex')
print()
print('q_variance_dominance_flag         :', sg.q_variance_dominance_flag)
print('share < s_be at every grid point  :', all(pt.fx_variance_share < S_BE for pt in sg.points))

=== §9 NON-RETIREMENT-rung input ===
panel-mean share (grid-invariant) : 1.535158e-04
s_be threshold                    : 0.25
1 - s_be (Q-dominance threshold)  : 0.75
share / s_be (decades below)      : -3.21 dex

q_variance_dominance_flag         : True
share < s_be at every grid point  : True


### Interpretation

`q_variance_dominance_flag = True` on the real panel. The panel-mean share (~1.5e-4) is approximately 3.2 decades below s_be = 0.25 at every grid point on the anchored Q-range. Equivalently, Q-variance accounts for >>75% of the cost-stream variance across the whole anchored range - the spec-anticipated NON-RETIREMENT-rung input to the §9 classifier.

This is the HALT-Q-DOMINANCE outcome spec §7 field 7b explicitly anticipates and accepts. It is NOT a failure; the methods-paper §5 anchor (Bhaduri-Laski-Riese 2006 / Minsky P_k/P_i / AMM micro-structure mapping) survives any §9 verdict because it is the framework's load-bearing contribution, independent of any single iteration's surface shape. Phase 6's §9 classifier will route on this flag.

## Trio 5 - interior-crossing scan under the user-locked rule (task 4.2a)

Why: the user-locked interior-crossing rule (2026-05-21) requires >=3 contiguous strictly-interior grid cells with share < s_be to flag a crossing - single-cell and 2-cell dips are dominated by simulator Monte Carlo noise and are excluded. Endpoint dips are reported separately (endpoint-violation, not interior-crossing). Critically, the rule also requires at least one endpoint to clear s_be: a surface entirely below s_be is `q_variance_dominance_flag = True`, NOT an interior crossing.

### Decision-citation block - user-locked interior-crossing rule

> Reference: pre-Phase-4 Model QA review item 3 (recommended tightening); user lock 2026-05-21; spec v0.7 §4.4 / plan task 4.2a / CORR-E10P-4.
> Why: single-cell and 2-cell dips are dominated by simulator MC noise per cell; a genuine interior crossing spans a region, not a point. The endpoint-clearance guard distinguishes 'surface entirely below threshold' (Q-dominance) from a non-monotone interior dip.
> Relevance: a noise-induced single-cell dip would otherwise flip the §9 `interior_crossing` flag and destabilize the verdict-rung classification.
> Connection: the §9 classifier consumes `interior_crossing` directly alongside `q_variance_dominance_flag`; the two flags route to disjoint rungs.

In [5]:
# Run the user-locked interior-crossing scan (min_run_length = 3).
ic = detect_interior_crossing(shares, S_BE)

print('=== task 4.2a interior-crossing scan (user-locked rule) ===')
print('interior_crossing       :', ic.interior_crossing)
print('endpoint_below_low      :', ic.endpoint_below_low)
print('endpoint_below_high     :', ic.endpoint_below_high)
print('qualifying interior runs:', ic.interior_runs)
print()
print('semantic disambiguation:')
print('  both endpoints below s_be   =>', ic.endpoint_below_low and ic.endpoint_below_high)
print('  q_variance_dominance_flag    =', sg.q_variance_dominance_flag)
print('  => surface entirely below threshold = Q-dominance, NOT interior-crossing')

=== task 4.2a interior-crossing scan (user-locked rule) ===
interior_crossing       : False
endpoint_below_low      : True
endpoint_below_high     : True
qualifying interior runs: ((1, 48),)

semantic disambiguation:
  both endpoints below s_be   => True
  q_variance_dominance_flag    = True
  => surface entirely below threshold = Q-dominance, NOT interior-crossing


### Interpretation

`interior_crossing = False` under the user-locked rule. Both endpoints have share < s_be (`endpoint_below_low = True`, `endpoint_below_high = True`), so by the user-locked semantic guard a 'crossing' cannot fire - a surface entirely below s_be is Q-dominance (already captured by `q_variance_dominance_flag = True`), NOT an interior-crossing. There are zero qualifying interior runs.

This is the correct disambiguation. A naive 'any below-threshold cell' rule (the legacy Phase-0 RED-harness variant) would falsely flag a crossing here; the user-locked rule routes the panel-wide below-threshold surface to the NON-RETIREMENT rung via the `q_variance_dominance_flag`, where it belongs.

## Trio 6 - non-statistical 5-currency min-max / IQR spread display (task 4.3)

Why: at G=5 the wild-cluster bootstrap is structurally size-broken (G well below the ~12 cluster threshold) and the permutation arm has 2^5 = 32-point support with EM/DM exchangeability violated - both bands were DROPPED per CORR-E10P-11 / spec v0.6 W-2. The replacement is the raw 5-currency min-max / IQR envelope: overplot the 5 per-currency surfaces and report the pointwise envelope across the 5 observed values. NOT a confidence interval, NOT an inferential band, NOT a hypothesis test - a descriptive spread of 5 observed currency values, FULL STOP. The display gates no verdict.

### Decision-citation block - non-statistical envelope replaces dropped bootstrap / permutation

> Reference: CORR-E10P-11 / spec v0.6 W-2 (bootstrap + permutation arms dropped at G=5); pre-Phase-4 Model QA review item 4 (in-figure firewall label + EM/DM cosmetic colour coding).
> Why: at G=5 the bootstrap is size-broken and the permutation arm's support is too small for a meaningful test; neither produces a defensible band. The raw 5-currency envelope is the honest descriptive substitute - a spread, not a band.
> Relevance: the spread display gates no verdict; it is descriptive content the methods-paper §5 anchor uses to motivate the convex hedge's cross-currency design.
> Connection: the in-figure firewall label is rendered as a figure annotation so a figure escaping its prose still self-firewalls.

In [6]:
# Compute the 5-currency non-statistical envelope.
cs = compute_currency_spread(cells_panel, q_range=Q_RANGE, s_be=S_BE)

env_q = list(cs.envelope_q_grid)
env_min = list(cs.envelope_min)
env_max = list(cs.envelope_max)
env_q1 = list(cs.envelope_q1)
env_q3 = list(cs.envelope_q3)
env_med = list(cs.envelope_median)

print('=== §4.3 non-statistical 5-currency spread ===')
print('grid points          :', len(env_q))
print('Q-range              :', (env_q[0], env_q[-1]))
print()
print('envelope at Q ~ {:.1f}:'.format(env_q[0]))
print('  min    :', format(env_min[0], '.6e'))
print('  Q1     :', format(env_q1[0], '.6e'))
print('  median :', format(env_med[0], '.6e'))
print('  Q3     :', format(env_q3[0], '.6e'))
print('  max    :', format(env_max[0], '.6e'))
print()
print('per-currency mean share (EM = COP, BRL, NGN; DM = EUR, GBP):')
for cur in sorted(cs.per_currency_grids):
    sh = cs.per_currency_grids[cur].points[0].fx_variance_share
    print('  {:3s} ({:2s}) :'.format(cur, cs.is_em_or_dm[cur]),
          format(sh, '.6e'))

# EM / DM colour palette: EM = warm tones, DM = cool tones.
EM_COLORS = {'COP': '#d62728', 'BRL': '#ff7f0e', 'NGN': '#bcbd22'}
DM_COLORS = {'EUR': '#1f77b4', 'GBP': '#17becf'}
COLORS = {**EM_COLORS, **DM_COLORS}

fig, ax = plt.subplots(figsize=(10, 6))
for cur, surf in cs.per_currency_grids.items():
    tag = cs.is_em_or_dm[cur]
    qq = [pt.q_volume for pt in surf.points]
    sh = [pt.fx_variance_share for pt in surf.points]
    ax.plot(qq, sh, color=COLORS[cur], linewidth=1.6,
            label='{} ({})'.format(cur, tag))

# Min-max envelope (light) + IQR band (darker).
ax.fill_between(env_q, env_min, env_max, color='grey', alpha=0.18,
                label='min-max envelope (5 currencies)')
ax.fill_between(env_q, env_q1, env_q3, color='grey', alpha=0.32,
                label='IQR (Q1-Q3 across 5 currencies)')
ax.plot(env_q, env_med, color='black', linewidth=1.5,
        linestyle='-.', label='median of 5 currencies')
ax.axhline(S_BE, color='red', linestyle='--', linewidth=1.5,
           label='s_be = 0.25 (Phase-2.5 frozen)')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('monthly Q-volume (log-scale)')
ax.set_ylabel('FX-variance share  Var(dlogFX) / Var(dlogcost)  (log-scale)')
ax.set_title('E10 5-currency spread - NON-STATISTICAL; descriptive only')

# Mandatory in-figure firewall label (per Model QA review item 4).
ax.text(
    0.02, 0.04,
    cs.non_statistical_label,
    transform=ax.transAxes,
    fontsize=9.5,
    color='darkred',
    bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
              edgecolor='darkred', alpha=0.95),
)
ax.legend(loc='center right', fontsize=8)
ax.grid(True, which='both', linestyle=':', alpha=0.4)
plt.tight_layout()
plt.show()

=== §4.3 non-statistical 5-currency spread ===
grid points          : 50
Q-range              : (28.0, 130.00000000000006)

envelope at Q ~ 28.0:
  min    : 2.100712e-05
  Q1     : 2.907963e-05
  median : 5.921762e-05
  Q3     : 6.159166e-05
  max    : 5.966828e-04

per-currency mean share (EM = COP, BRL, NGN; DM = EUR, GBP):
  BRL (EM) : 5.921762e-05
  COP (EM) : 6.159166e-05
  EUR (DM) : 2.907963e-05
  GBP (DM) : 2.100712e-05
  NGN (EM) : 5.966828e-04


/tmp/ipykernel_64677/1855650279.py:70: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Interpretation

The envelope at Q ~ 28 spans min = 2.10e-05 (EUR/GBP DM currencies, the lowest FX-vol) to max = 5.97e-04 (NGN EM, the highest FX-vol regime including the post-NFEM-float naira) with median = 5.92e-05. Every per-currency surface and the entire envelope sit far below s_be = 0.25 (~3 decades below at the median). EM currencies (COP, BRL, NGN) carry systematically higher shares than DM currencies (EUR, GBP) - the expected descriptive cross-currency ordering.

The spread is rendered with the mandatory in-figure firewall label: '5-currency min-max / IQR -- NOT a confidence interval; NOT an inferential band; NOT a hypothesis test.' Per CORR-E10P-11 / spec v0.6 W-2 this raw 5-currency display replaces the dropped bootstrap / permutation bands and gates no verdict.

## Trio 7 - surface vs break-even comparison

Why: the spec §9 classifier reads the qualitative relationship between the calibration-conditional surface and the ex-ante-pinned break-even share. This trio overlays the panel-mean surface, the 5-currency envelope, and the s_be = 0.25 reference line on a single figure so the descriptive comparison is exhibited in one place. The figure is descriptive only; it gates no verdict.

### Decision-citation block - one-figure descriptive comparison

> Reference: spec v0.7 §4.3 / §9 (surface is the deliverable; s_be is the ex-ante threshold).
> Why: the §9 classifier rung selection is more legible when the panel-mean surface and the 5-currency envelope are read against the same s_be reference simultaneously; one figure beats three.
> Relevance: the overlay makes the 3-decade gap between every observed share and s_be visible at a glance - which is what the `q_variance_dominance_flag = True` numerically asserts.
> Connection: feeds the closing trio's §9-classifier-input summary.

In [7]:
# Combined descriptive overlay: panel-mean + envelope + s_be reference.
fig, ax = plt.subplots(figsize=(10, 6))

ax.fill_between(env_q, env_min, env_max, color='grey', alpha=0.18,
                label='5-currency min-max envelope (NON-STATISTICAL)')
ax.fill_between(env_q, env_q1, env_q3, color='grey', alpha=0.32,
                label='5-currency IQR (NON-STATISTICAL)')
ax.plot(env_q, env_med, color='black', linestyle='-.', linewidth=1.4,
        label='5-currency median')
ax.plot(q_grid, shares, color='black', linewidth=2.5,
        label='panel-mean surface')
ax.axhline(S_BE, color='red', linestyle='--', linewidth=1.8,
           label='s_be = 0.25 (Phase-2.5 frozen)')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('monthly Q-volume (log-scale)')
ax.set_ylabel('FX-variance share  (log-scale)')
ax.set_title('E10 surface vs break-even - DESCRIPTIVE; calibration-conditional')
ax.text(
    0.02, 0.04,
    'DESCRIPTIVE overlay - the entire surface and the 5-currency '
    'envelope sit ~3 dex below s_be everywhere on the anchored '
    'Q-range. NOT an inferential band; NOT a hypothesis test.',
    transform=ax.transAxes,
    fontsize=9,
    color='darkred',
    bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
              edgecolor='darkred', alpha=0.95),
    wrap=True,
)
ax.legend(loc='center right', fontsize=8)
ax.grid(True, which='both', linestyle=':', alpha=0.4)
plt.tight_layout()
plt.show()

# Distance from s_be in decades.
panel_mean_dex = math.log10(shares[0] / S_BE)
env_max_dex = math.log10(max(env_max) / S_BE)
env_min_dex = math.log10(min(env_min) / S_BE)
print('decades below s_be (negative = below):')
print('  panel-mean surface   :', format(panel_mean_dex, '.2f'), 'dex')
print('  envelope max (NGN)   :', format(env_max_dex, '.2f'), 'dex')
print('  envelope min (DM)    :', format(env_min_dex, '.2f'), 'dex')

decades below s_be (negative = below):
  panel-mean surface   : -3.21 dex
  envelope max (NGN)   : -2.62 dex
  envelope min (DM)    : -4.08 dex


/tmp/ipykernel_64677/3986974518.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Interpretation

The entire panel-mean surface and the entire 5-currency envelope sit approximately 2.6-4.1 decades below s_be = 0.25 everywhere on the anchored Q-range. The envelope maximum (NGN, the EM currency with the highest FX-vol regime) is ~2.6 decades below s_be; the panel-mean surface is ~3.2 decades below; the envelope minimum (DM) is ~4.1 decades below.

Phase 6's §9 classifier (plan task 6.1) will consume `q_variance_dominance_flag = True` and route this iteration to NON-RETIREMENT - a valid, acceptable, spec-anticipated outcome per §7 field 7b. The methods-paper §5 anchor (Bhaduri-Laski-Riese 2006 / Minsky P_k/P_i / AMM micro-structure mapping) survives any §9 outcome because it is the framework's load-bearing contribution, independent of any single iteration's surface shape. No fishing - the descriptive content read here is the honest output of honest computation against an ex-ante-frozen threshold.

## Trio 8 - closing summary: the four §9-classifier inputs handed to Phase 6

Why: Phase 4 (task 4.4) produces four boolean flags the Phase-6 §9 classifier (task 6.1) consumes. This trio collects the flags and the descriptive numbers in one place so the handoff to Phase 6 is explicit and reproducible.

### Decision-citation block - the four §9 inputs

> Reference: spec v0.7 §9 (descriptive verdict ladder); plan v0.2 task 6.1 (§9 classifier).
> Why: the §9 classifier is a deterministic boolean-routing step - the four flags fully determine the rung selection; collecting them in one cell makes the routing inspectable.
> Relevance: the four flags are the entire Phase 4 -> Phase 6 interface; if any flag changes between this notebook and the Phase-6 classifier the §9 rung changes deterministically.
> Connection: closes Phase 4 (task 4.4) and opens Phase 6 (task 6.1).

In [8]:
# Collect the four §9-classifier inputs.
surface_computed_flag = True  # the §4.3 surface evaluator ran without HALT
material_gap_flag = any(c.material_gap for c in cells_panel)
q_variance_dominance_flag = sg.q_variance_dominance_flag
interior_crossing_flag = ic.interior_crossing

summary = {
    'phase': 'E10 GSPS Phase 4 / task 4.4',
    'panel_cells': len(cells_panel),
    'currencies': sorted(by_cur),
    'anchored_q_range': list(Q_RANGE),
    'frozen_s_be': S_BE,
    'effective_n_grid': sg.effective_n_grid,
    'refined': sg.refined,
    'panel_mean_share': float(shares[0]),
    'envelope_at_q_low': {
        'min': float(env_min[0]),
        'q1': float(env_q1[0]),
        'median': float(env_med[0]),
        'q3': float(env_q3[0]),
        'max': float(env_max[0]),
    },
    'vol_on_vol': {
        'beta_two_way': vov.beta_two_way,
        'beta_currency_only': vov.beta_currency_only,
        'gap': vov.gap,
        'sign_two_way': vov.sign_two_way,
        'sign_currency_only': vov.sign_currency_only,
    },
    'section_9_inputs': {
        'surface_computed_flag': surface_computed_flag,
        'material_gap_flag': material_gap_flag,
        'q_variance_dominance_flag': q_variance_dominance_flag,
        'interior_crossing_flag': interior_crossing_flag,
    },
    'trajectory': 'NON-RETIREMENT (spec §7 field 7b; §9 NON-RETIREMENT rung)',
}

print(json.dumps(summary, indent=2, default=float))

# Persist the cross-cell-reproducibility artifact.
out_path = REPO / 'notebooks' / 'e10_gsps' / 'diagnostics' / 'E10.3_surface_summary.json'
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(summary, indent=2, default=float))
print()
print('wrote', out_path.relative_to(REPO))

{
  "phase": "E10 GSPS Phase 4 / task 4.4",
  "panel_cells": 150,
  "currencies": [
    "BRL",
    "COP",
    "EUR",
    "GBP",
    "NGN"
  ],
  "anchored_q_range": [
    28.0,
    130.0
  ],
  "frozen_s_be": 0.25,
  "effective_n_grid": 50,
  "refined": false,
  "panel_mean_share": 0.0001535157617749653,
  "envelope_at_q_low": {
    "min": 2.1007120777926055e-05,
    "q1": 2.907963130407722e-05,
    "median": 5.921762100760453e-05,
    "q3": 6.15916596650237e-05,
    "max": 0.000596682776120195
  },
  "vol_on_vol": {
    "beta_two_way": 68.15124620607666,
    "beta_currency_only": 54.714823014552536,
    "gap": 13.436423191524128,
    "sign_two_way": 1,
    "sign_currency_only": 1
  },
  "section_9_inputs": {
    "surface_computed_flag": true,
    "material_gap_flag": false,
    "q_variance_dominance_flag": true,
    "interior_crossing_flag": false
  },
  "trajectory": "NON-RETIREMENT (spec \u00a77 field 7b; \u00a79 NON-RETIREMENT rung)"
}

wrote notebooks/e10_gsps/diagnostics/E10.3_su

### Interpretation

The four §9-classifier inputs the Phase-6 verdict step (plan task 6.1) consumes from this notebook:

- `surface_computed_flag = True` - the §4.3 surface evaluator ran cleanly on the full anchored Q-range, no HALT-SURFACE-UNCOMPUTABLE.
- `material_gap_flag = False` - the 150-cell panel has zero material-gap cells (every cell has >=2 surviving days for the decomposition).
- `q_variance_dominance_flag = True` - the panel-mean share is ~3 decades below s_be at every grid point; Q-variance dominates panel-wide. This is the spec §7 field 7b acceptable outcome.
- `interior_crossing_flag = False` - correct under the user-locked rule; the surface is entirely below s_be (Q-dominance), not non-monotone (which would have flagged an interior crossing).

Trajectory: NON-RETIREMENT. The §9 classifier in Phase 6 will route this iteration to the NON-RETIREMENT rung per spec §7 field 7b and §9 ladder semantics - the descriptive outcome is valid and acceptable; no spec-vs-data contradiction, no anti-fishing HALT. The methods-paper §5 anchor survives any §9 outcome. The four flags + the descriptive numbers above are persisted to `diagnostics/E10.3_surface_summary.json` for Phase-6 cross-cell reproducibility.

## Summary - Phase 4 (E10.3) surface characterization

- §4.1 vol-on-vol descriptive sign gate: both co-primary FE specs return sign +1 (beta_two_way = +68.151246, beta_currency_only = +54.714823); Frisch-Waugh-Lovell gap = +13.436423 descriptive content load-bearing for the §13 M-sketch.
- §4.3 FX-variance-share surface evaluated on the user-locked log-spaced 50-point grid across Q in [28, 130]; adaptive doubling did not fire; panel-mean share ~1.5e-4 grid-invariant.
- `q_variance_dominance_flag = True` (share ~3 decades below s_be at every grid point) - spec §7 field 7b acceptable outcome.
- `interior_crossing = False` (correct under user-locked >=3-contiguous rule; both endpoints below s_be => Q-dominance, NOT a crossing).
- 5-currency non-statistical envelope at Q ~ 28: min 2.10e-05 (EUR/GBP DM), max 5.97e-04 (NGN EM), median 5.92e-05; all sit far below s_be.
- Diagnostics: `diagnostics/E10.3_surface_summary.json` (the four §9-classifier flags + envelope summary, written by the closing trio).
- Trajectory: NON-RETIREMENT.